# 📏 Context Window Management

## The Problem

LLMs have a **context window limit** — the maximum number of tokens they can process at once.

| Model | Context window |
|---|---|
| GPT-3.5 | 16k tokens |
| GPT-4o | 128k tokens |
| Claude Sonnet | 200k tokens |

In RAG, you retrieve N documents. If they're too long, they won't fit.

```
Context window: 4000 tokens
Each document: ~800 tokens
You retrieved: 10 documents

Total: 8000 tokens → OVERFLOW ❌
Fits:  5 documents → 4000 tokens ✅
```

## Strategies We'll Cover

1. **Count tokens** before sending
2. **Truncate documents** that are too long
3. **Rank and keep the best** docs within the budget
4. **Lost in the middle** — why order matters

In [ ]:
# We'll estimate tokens simply: ~4 characters per token (rough but useful)
# For exact counts use: tiktoken (OpenAI) or anthropic's token counter

def count_tokens(text):
    """Rough token estimate: 1 token ≈ 4 characters"""
    return len(text) // 4

# Test
sample = "Retrieval-Augmented Generation combines retrieval with LLM generation."
print(f"Text: '{sample}'")
print(f"Estimated tokens: {count_tokens(sample)}")
print(f"Actual characters: {len(sample)}")

## 1. Check If Docs Fit

In [ ]:
# Simulate retrieved documents of varying lengths
retrieved_docs = [
    {"id": 0, "score": 0.92, "text": "RAG combines retrieval with LLM generation to produce grounded answers."},
    {"id": 1, "score": 0.88, "text": "The retriever uses embedding similarity to find relevant documents from a large corpus. "
                                      "Each document is encoded into a dense vector, and the query vector is compared using cosine similarity. "
                                      "The top-k most similar documents are returned as candidates for the generation step."},
    {"id": 2, "score": 0.85, "text": "Context is injected into the prompt between the system instruction and the user question."},
    {"id": 3, "score": 0.81, "text": "Vector databases like Pinecone, Weaviate, and Chroma store precomputed embeddings and support "
                                      "fast approximate nearest-neighbor search. They are optimized for high-dimensional vector operations "
                                      "and can handle millions of documents efficiently. Most support metadata filtering as well, "
                                      "allowing you to narrow the search space before running similarity search."},
    {"id": 4, "score": 0.77, "text": "Chunking splits long documents into smaller pieces for better retrieval granularity."},
]

TOKEN_BUDGET = 200  # Small budget to demonstrate overflow

print(f"Token budget: {TOKEN_BUDGET}\n")
print(f"{'Doc':>4} {'Score':>6} {'Tokens':>7}  Text preview")
print("-" * 70)

total = 0
for d in retrieved_docs:
    t = count_tokens(d["text"])
    total += t
    status = "✅" if total <= TOKEN_BUDGET else "❌ overflow"
    print(f"  {d['id']:>2} {d['score']:>6.2f} {t:>7}  {d['text'][:45]}... {status}")

print(f"\nTotal tokens: {total} / {TOKEN_BUDGET} budget")

## 2. Strategy 1 — Keep Top Docs Within Budget

In [ ]:
# Sort by relevance score (best first), include docs until we hit the budget

def select_docs_by_budget(docs, token_budget):
    """
    Greedily include docs from highest to lowest score
    until we run out of token budget.
    """
    selected = []
    used_tokens = 0

    # Docs are already sorted by score (highest first)
    for doc in docs:
        doc_tokens = count_tokens(doc["text"])

        if used_tokens + doc_tokens <= token_budget:
            selected.append(doc)
            used_tokens += doc_tokens
        else:
            print(f"  Skipping doc {doc['id']} ({doc_tokens} tokens) — would exceed budget")

    return selected, used_tokens


selected, used = select_docs_by_budget(retrieved_docs, TOKEN_BUDGET)

print(f"\nSelected {len(selected)} / {len(retrieved_docs)} docs")
print(f"Tokens used: {used} / {TOKEN_BUDGET}")
print("\nSelected docs:")
for d in selected:
    print(f"  [{d['score']:.2f}] {d['text'][:65]}...")

## 3. Strategy 2 — Truncate Long Documents

In [ ]:
# Instead of skipping a doc entirely, truncate it to fit

def truncate_doc(text, max_tokens):
    """
    Truncate text to fit within max_tokens.
    Cuts at a word boundary to avoid broken words.
    """
    max_chars = max_tokens * 4  # Convert tokens back to chars
    if len(text) <= max_chars:
        return text
    # Cut at last space before limit
    truncated = text[:max_chars]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] + "..."


# Apply: allocate equal token budget per doc
tokens_per_doc = TOKEN_BUDGET // len(retrieved_docs)

print(f"Token budget: {TOKEN_BUDGET}")
print(f"Tokens per doc: {tokens_per_doc}\n")

truncated_docs = []
for d in retrieved_docs:
    truncated_text = truncate_doc(d["text"], tokens_per_doc)
    truncated_docs.append({**d, "text": truncated_text})
    before = count_tokens(d["text"])
    after = count_tokens(truncated_text)
    print(f"Doc {d['id']}: {before} tokens → {after} tokens")
    print(f"  {truncated_text[:80]}")
    print()

total_after = sum(count_tokens(d["text"]) for d in truncated_docs)
print(f"Total tokens after truncation: {total_after} / {TOKEN_BUDGET}")

## 4. The "Lost in the Middle" Problem

Research shows LLMs pay most attention to the **beginning and end** of the context window.  
Documents in the **middle** are often ignored.

**Fix: Put the most relevant document first.**

In [ ]:
# Wrong order — most relevant doc is in the middle
bad_order = [
    retrieved_docs[4],  # score 0.77 — least relevant first
    retrieved_docs[3],  # score 0.81
    retrieved_docs[0],  # score 0.92 ← most relevant, buried in the middle!
    retrieved_docs[2],  # score 0.85
    retrieved_docs[1],  # score 0.88
]

# Correct order — best docs first
good_order = sorted(retrieved_docs, key=lambda d: d["score"], reverse=True)

print("❌ Bad order (random / reverse):")
for d in bad_order:
    print(f"   [{d['score']:.2f}] {d['text'][:60]}...")

print("\n✅ Good order (best relevance first):")
for d in good_order:
    print(f"   [{d['score']:.2f}] {d['text'][:60]}...")

In [ ]:
# Build the final context block — best doc first, within token budget

def build_context(docs, token_budget):
    """Build a context string from docs, sorted by relevance, within token budget."""
    # Sort best first
    sorted_docs = sorted(docs, key=lambda d: d["score"], reverse=True)

    context_parts = []
    used = 0

    for i, doc in enumerate(sorted_docs, 1):
        doc_tokens = count_tokens(doc["text"])
        if used + doc_tokens > token_budget:
            break
        context_parts.append(f"[{i}] {doc['text']}")
        used += doc_tokens

    return "\n".join(context_parts), used


context, tokens_used = build_context(retrieved_docs, TOKEN_BUDGET)
print(f"Context built: {tokens_used} / {TOKEN_BUDGET} tokens\n")
print(context)

## Key Takeaways 🎯

1. **Always count tokens** before sending — don't assume it fits
2. **Greedy selection**: take best docs until budget is full
3. **Truncate** long docs rather than drop them entirely (preserves partial signal)
4. **Order matters** — best docs first (LLMs attend more to the beginning)
5. **Context ≠ more is better** — irrelevant docs hurt performance

**Practical rule:** Start with top 3–5 docs. Increase only if you measure improvement.

---
Next: `03_Grounding_and_Citations.ipynb` — make the LLM cite its sources reliably.